# LLaMA 3.1 Fine-Tuning: Unsloth + Classification Head + LoRA HPT + Staged Training

## Architecture
```
LLaMA 3.1 8B (frozen, 4-bit via Unsloth)
    + LoRA Adapters (trainable) ← Unsloth get_peft_model
    + Classification Head (trainable) ← Linear(4096 → 512 → 10)
```

## Strategy
- Same 80/10/10 split as Classical ML (random_state=42, stratify)
- Staged training: 8 stages × 3000 = 24,000 samples
- LoRA HPT: 3 configs compared (r=8, r=16, r=32)
- Checkpoints saved to Google Drive after every stage
- Full evaluation (accuracy, F1, confusion matrix) after every stage

## How to Run
- **First time**: Run Cells 1→10, then Cell 11
- **Resume session**: Run Cells 1→10, then Cell 12
- **Final results**: Cell 13 (test set) + Cell 14 (compare all configs)

## ⚙️ Cell 1 — Install Libraries

In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if "llama" in file.lower() and file.endswith(".ipynb"):
            print(os.path.join(root, file))

In [ ]:
import json
from pathlib import Path
from google.colab import drive, files

drive.mount('/content/drive')

old_path = Path("/content/drive/MyDrive/Colab Notebooks/llama_finetune31_8B.ipynb")
new_path = Path("/content/drive/MyDrive/Colab Notebooks/llama_finetune31_8B.ipynb")

nb = json.loads(old_path.read_text(encoding="utf-8"))

# امسح widgets من metadata الرئيسية
nb.get("metadata", {}).pop("widgets", None)

# امسح outputs كمان احتياطيًا
for cell in nb.get("cells", []):
    cell["outputs"] = []
    cell["execution_count"] = None

new_path.write_text(json.dumps(nb, indent=1, ensure_ascii=False), encoding="utf-8")

# تحقق
check = json.loads(new_path.read_text(encoding="utf-8"))
print("widgets exists?", "widgets" in check.get("metadata", {}))
print("Saved fixed file:", new_path)

files.download(str(new_path))

In [ ]:
# Run once per session
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q transformers datasets scikit-learn
print('✅ Installation done')

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q "transformers>=4.45.0,<5.0.0"  # ← قيّد الـ version
!pip install -q datasets scikit-learn
print('✅ Installation done')

## 📦 Cell 2 — Imports

In [ ]:
import os
import gc
import json
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix
)
from torch.utils.data import Dataset, DataLoader
from transformers import get_cosine_schedule_with_warmup

# ── Unsloth (loads LLaMA fast + 4-bit quantization)
from unsloth import FastLanguageModel
from peft import PeftModel

print('✅ All imports done')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU — check runtime"}')

## 🔧 Cell 3 — Config

In [ ]:
# ════════════════════════════════════════════
# PATHS
# ════════════════════════════════════════════
DATA_PATH       = '/content/drive/MyDrive/IT Support Ticket Data.csv'
CHECKPOINT_BASE = '/content/drive/MyDrive/llama-cls-checkpoints'

# ════════════════════════════════════════════
# MODEL
# ════════════════════════════════════════════
MODEL_NAME  = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'
MAX_SEQ_LEN = 512

# ════════════════════════════════════════════
# DATA SPLIT — identical to Classical ML
# ════════════════════════════════════════════
RANDOM_STATE = 42
TEST_SIZE    = 0.2   # 80% train, 20% temp
VAL_SIZE     = 0.5   # temp → 10% val + 10% test

# ════════════════════════════════════════════
# STAGED TRAINING
# ════════════════════════════════════════════
STAGE_SIZE = 3000   # samples per stage
NUM_STAGES = 7      # 7 × 3000 = 21,000 ≈ full train set (~23,720)

# ════════════════════════════════════════════
# TRAINING HYPERPARAMETERS
# ════════════════════════════════════════════
BATCH_SIZE   = 2    # small to avoid OOM on Colab free GPU
GRAD_ACCUM   = 16   # effective batch = 2 × 16 = 32
NUM_EPOCHS   = 3    # per stage
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

# ════════════════════════════════════════════
# LORA — target only q_proj & v_proj
# (most impactful for classification, less memory)
# ════════════════════════════════════════════
LORA_TARGET_MODULES = ['q_proj', 'v_proj', 'k_proj', 'o_proj']

# LoRA Hyperparameter Tuning configs
# Rule: lora_alpha = r × 2  (proven best ratio in research)
LORA_CONFIGS = [
    {'name': 'LoRA_A', 'r': 8,  'lora_alpha': 16, 'lr': 1e-4},  # conservative baseline
    {'name': 'LoRA_B', 'r': 16, 'lora_alpha': 32, 'lr': 2e-4},  # standard ← start here
    {'name': 'LoRA_C', 'r': 32, 'lora_alpha': 64, 'lr': 2e-4},  # aggressive
]

# ════════════════════════════════════════════
# CLASSES
# ════════════════════════════════════════════
DEPARTMENTS = [
    'Billing and Payments',
    'Customer Service',
    'General Inquiry',
    'Human Resources',
    'IT Support',
    'Product Support',
    'Returns and Exchanges',
    'Sales and Pre-Sales',
    'Service Outages and Maintenance',
    'Technical Support'
]
NUM_CLASSES = len(DEPARTMENTS)
LABEL2ID    = {d: i for i, d in enumerate(DEPARTMENTS)}
ID2LABEL    = {i: d for i, d in enumerate(DEPARTMENTS)}

print('✅ Config loaded')
print(f'   Model        : {MODEL_NAME}')
print(f'   Stages       : {NUM_STAGES} × {STAGE_SIZE} = {NUM_STAGES*STAGE_SIZE} samples')
print(f'   LoRA configs : {[c["name"] for c in LORA_CONFIGS]}')
print(f'   Batch (eff.) : {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM}')
print(f'   Checkpoints  : {CHECKPOINT_BASE}')

## 💾 Cell 4 — Mount Drive & Load Data

In [ ]:
drive.mount('/content/drive')
os.makedirs(CHECKPOINT_BASE, exist_ok=True)

df = pd.read_csv(DATA_PATH, index_col=0)
df = df.dropna(subset=['Body']).reset_index(drop=True)
df = df[['Body', 'Department']]

print(f'Total samples : {len(df)}')
print(f'Departments   : {df["Department"].nunique()}')
print()
print(df['Department'].value_counts())

## ✂️ Cell 5 — Data Split (Same as Classical ML)

In [ ]:
# ════════════════════════════════════════════
# EXACT same logic as Classical ML notebook:
#   Step 1 → 80% train / 20% temp  (stratified, seed=42)
#   Step 2 → temp: 50% val / 50% test  (stratified, seed=42)
# ════════════════════════════════════════════

train_df, temp_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    stratify=df['Department'],
    random_state=RANDOM_STATE
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=VAL_SIZE,
    stratify=temp_df['Department'],
    random_state=RANDOM_STATE
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

# ── Build stage slices (non-overlapping chunks from train_df)
total_needed = NUM_STAGES * STAGE_SIZE
assert len(train_df) >= total_needed, (
    f'Need {total_needed} train rows but only have {len(train_df)}. '
    f'Reduce NUM_STAGES or STAGE_SIZE in Cell 3.'
)

stages = [
    train_df.iloc[i*STAGE_SIZE : (i+1)*STAGE_SIZE].reset_index(drop=True)
    for i in range(NUM_STAGES)
]

print('=' * 55)
print('  DATA SPLIT SUMMARY')
print('=' * 55)
print(f'  Train : {len(train_df):>6}  (80%)')
print(f'  Val   : {len(val_df):>6}  (10%)')
print(f'  Test  : {len(test_df):>6}  (10%)')
print(f'  Stages: {NUM_STAGES} × {STAGE_SIZE} = {total_needed} used from train')
print()
for i, s in enumerate(stages):
    print(f'  Stage {i+1}: rows {i*STAGE_SIZE}–{(i+1)*STAGE_SIZE-1} | '
          f'min_dept={s["Department"].value_counts().min()} '
          f'max_dept={s["Department"].value_counts().max()}')

## 🔤 Cell 6 — Load Tokenizer & Build Datasets

In [ ]:
# ── Load tokenizer via Unsloth (same call used in Cell 7 for the model)
# We load model+tokenizer here once and reuse them everywhere
_, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

# ── PyTorch Dataset
class TicketDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.texts     = dataframe['Body'].astype(str).tolist()
        self.labels    = [LABEL2ID[d] for d in dataframe['Department'].tolist()]
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ── Build val & test loaders once (reused across all stages & configs)
val_dataset  = TicketDataset(val_df,  tokenizer, MAX_SEQ_LEN)
test_dataset = TicketDataset(test_df, tokenizer, MAX_SEQ_LEN)
val_loader   = DataLoader(val_dataset,  batch_size=BATCH_SIZE*2, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE*2, shuffle=False)

print(f'✅ Tokenizer ready')
print(f'   Val  batches : {len(val_loader)}')
print(f'   Test batches : {len(test_loader)}')

## 🏗️ Cell 7 — Model: LLaMA + Unsloth LoRA + Classification Head

In [ ]:
class LlamaClassifier(nn.Module):
    """
    LLaMA 3.1 8B loaded via Unsloth (4-bit, frozen)
        + LoRA adapters via FastLanguageModel.get_peft_model (trainable)
        + Classification Head: Linear(4096→512→10) (trainable)

    Forward pass:
        input_ids, attention_mask
            → LLaMA hidden states  (B, L, 4096)
            → mean pooling         (B, 4096)
            → classifier           (B, 10)
            → cross-entropy loss
    """

    def __init__(self, model_name, num_classes, lora_cfg):
        super().__init__()

        # ── Step 1: Load LLaMA base via Unsloth
        # Unsloth handles: 4-bit quantization, flash attention, memory optimisation
        base_model, _ = FastLanguageModel.from_pretrained(
            model_name=model_name,
            max_seq_length=MAX_SEQ_LEN,
            dtype=None,          # auto (float16 on GPU)
            load_in_4bit=True,   # 4-bit NF4 quantization → fits Colab GPU
        )

        # ── Step 2: Add LoRA adapters via Unsloth
        # Only q_proj & v_proj are targeted (best for classification, less memory)
        self.backbone = FastLanguageModel.get_peft_model(
            base_model,
            r=lora_cfg['r'],
            lora_alpha=lora_cfg['lora_alpha'],
            target_modules=LORA_TARGET_MODULES,
            lora_dropout=0.05,
            bias='none',
            use_gradient_checkpointing='unsloth',  # Unsloth optimised checkpointing
            random_state=RANDOM_STATE,
        )
        self.backbone.print_trainable_parameters()

        # ── Step 3: Classification Head
        # hidden_size = 4096 for LLaMA 8B
        hidden_size = self.backbone.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes)
        ).to(self.backbone.device)

        # Xavier init for stable training
        for layer in self.classifier:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, input_ids, attention_mask, labels=None):
        # Get hidden states from LLaMA backbone
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )

        # Mean pooling over non-padding tokens only
        hidden = outputs.hidden_states[-1]           # (B, L, H)
        mask   = attention_mask.unsqueeze(-1).float()
        pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)  # (B, H)
        pooled = pooled.to(torch.float32)            # cast for the Linear layer

        logits = self.classifier(pooled)             # (B, 10)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {'loss': loss, 'logits': logits}


def build_model(lora_cfg):
    """Build a fresh LlamaClassifier for the given LoRA config."""
    # Clear GPU memory before loading
    gc.collect()
    torch.cuda.empty_cache()
    return LlamaClassifier(MODEL_NAME, NUM_CLASSES, lora_cfg)


print('✅ LlamaClassifier class defined')

## 📊 Cell 8 — Evaluation Function

In [ ]:
def evaluate(model, loader, split_name='Val', show_confusion=True):
    """
    Full evaluation on any DataLoader.
    Returns dict: accuracy, f1_macro, f1_weighted
    """
    model.eval()
    device     = next(model.parameters()).device
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels']

            out   = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = out['logits'].argmax(dim=-1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1m = f1_score(all_labels, all_preds, average='macro',    zero_division=0)
    f1w = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

    print('=' * 58)
    print(f'  {split_name}')
    print('=' * 58)
    print(f'  Accuracy    : {acc:.4f}')
    print(f'  F1 Macro    : {f1m:.4f}')
    print(f'  F1 Weighted : {f1w:.4f}')
    print()
    print(classification_report(
        all_labels, all_preds,
        target_names=DEPARTMENTS,
        zero_division=0
    ))

    if show_confusion:
        cm = confusion_matrix(all_labels, all_preds)
        plt.figure(figsize=(12, 10))
        sns.heatmap(
            cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[d[:14] for d in DEPARTMENTS],
            yticklabels=[d[:14] for d in DEPARTMENTS]
        )
        plt.title(f'Confusion Matrix — {split_name}', fontsize=13)
        plt.xlabel('Predicted'); plt.ylabel('True')
        plt.tight_layout(); plt.show()

    model.train()
    return {'accuracy': acc, 'f1_macro': f1m, 'f1_weighted': f1w}


print('✅ evaluate() defined')

## 💾 Cell 9 — Checkpoint Functions

In [ ]:
def save_checkpoint(model, optimizer, scheduler, stage, lora_name, metrics):
    """
    Save to: CHECKPOINT_BASE / lora_name / stage_N /
        lora_adapters/   ← LoRA weights (Unsloth format)
        classifier_head.pt
        training_state.pt  ← optimizer + scheduler
        metrics.json
    """
    ckpt_dir = os.path.join(CHECKPOINT_BASE, lora_name, f'stage_{stage}')
    os.makedirs(ckpt_dir, exist_ok=True)

    # 1. LoRA adapters (Unsloth / PEFT format)
    model.backbone.save_pretrained(os.path.join(ckpt_dir, 'lora_adapters'))

    # 2. Classification head
    torch.save(
        model.classifier.state_dict(),
        os.path.join(ckpt_dir, 'classifier_head.pt')
    )

    # 3. Optimizer + scheduler state
    torch.save({
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict() if scheduler else None,
        'stage':     stage,
        'metrics':   metrics
    }, os.path.join(ckpt_dir, 'training_state.pt'))

    # 4. Human-readable metrics
    with open(os.path.join(ckpt_dir, 'metrics.json'), 'w') as f:
        json.dump({'stage': stage, 'lora_config': lora_name, **metrics}, f, indent=2)

    print(f'  ✅ Checkpoint saved → {ckpt_dir}')
    print(f'     acc={metrics["accuracy"]:.4f}  F1-macro={metrics["f1_macro"]:.4f}')


def load_checkpoint(model, optimizer, scheduler, stage, lora_name):
    """
    Load LoRA adapters + classifier head + optimizer from a saved checkpoint.
    Returns True if successful, False if checkpoint not found.
    """
    ckpt_dir = os.path.join(CHECKPOINT_BASE, lora_name, f'stage_{stage}')
    if not os.path.exists(ckpt_dir):
        print(f'  ❌ Not found: {ckpt_dir}')
        return False

    # 1. Load LoRA adapters
    model.backbone = PeftModel.from_pretrained(
        model.backbone,
        os.path.join(ckpt_dir, 'lora_adapters'),
        is_trainable=True
    )

    # 2. Load classifier head
    model.classifier.load_state_dict(
        torch.load(
            os.path.join(ckpt_dir, 'classifier_head.pt'),
            map_location='cpu'
        )
    )

    # 3. Load optimizer + scheduler
    state = torch.load(
        os.path.join(ckpt_dir, 'training_state.pt'),
        map_location='cpu'
    )
    optimizer.load_state_dict(state['optimizer'])
    if scheduler and state.get('scheduler'):
        scheduler.load_state_dict(state['scheduler'])

    print(f'  ✅ Checkpoint loaded ← {ckpt_dir}')
    return True


def list_checkpoints():
    """Print all saved checkpoints and their metrics."""
    if not os.path.exists(CHECKPOINT_BASE):
        print('No checkpoints saved yet.')
        return
    print('=' * 60)
    print('  SAVED CHECKPOINTS')
    print('=' * 60)
    for lora_name in sorted(os.listdir(CHECKPOINT_BASE)):
        ldir = os.path.join(CHECKPOINT_BASE, lora_name)
        if not os.path.isdir(ldir): continue
        print(f'\n  📁 {lora_name}')
        for sdir in sorted(os.listdir(ldir)):
            mf = os.path.join(ldir, sdir, 'metrics.json')
            if os.path.exists(mf):
                with open(mf) as f: m = json.load(f)
                print(f'     {sdir}: acc={m.get("accuracy","?"):.4f}  '
                      f'F1-macro={m.get("f1_macro","?"):.4f}')


print('✅ Checkpoint functions defined')

## 🏋️ Cell 10 — Training Loop (One Stage)

In [ ]:
def train_one_stage(model, stage_df, optimizer, scheduler, stage_num, lora_name):
    """
    Train on one stage for NUM_EPOCHS epochs.
    Evaluate on val after each epoch.
    Save checkpoint at end of stage.
    """
    device       = next(model.parameters()).device
    stage_ds     = TicketDataset(stage_df, tokenizer, MAX_SEQ_LEN)
    train_loader = DataLoader(stage_ds, batch_size=BATCH_SIZE, shuffle=True)

    print(f'\n{"="*58}')
    print(f'  STAGE {stage_num} / {NUM_STAGES}  —  {lora_name}')
    print(f'  Samples : {len(stage_df)} | Batches : {len(train_loader)} | Epochs : {NUM_EPOCHS}')
    print(f'{"="*58}')

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        correct    = 0
        total      = 0
        optimizer.zero_grad()

        for step, batch in enumerate(train_loader):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            out  = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = out['loss'] / GRAD_ACCUM
            loss.backward()

            if (step + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0
                )
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            total_loss += out['loss'].item()
            preds       = out['logits'].argmax(dim=-1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)

            if (step + 1) % 100 == 0:
                print(f'   Epoch {epoch} | Step {step+1:>4}/{len(train_loader)} | '
                      f'Loss: {total_loss/(step+1):.4f} | '
                      f'Train Acc: {correct/total:.4f}')

        avg_loss = total_loss / len(train_loader)
        print(f'\n  ── Epoch {epoch} done: loss={avg_loss:.4f}  train_acc={correct/total:.4f}')

        # Evaluate on val after every epoch (no confusion matrix to save time)
        evaluate(
            model, val_loader,
            split_name=f'Val  [Stage {stage_num} | Epoch {epoch}]',
            show_confusion=False
        )

    # Full evaluation with confusion matrix at end of stage
    print(f'\n  ── Full evaluation at end of Stage {stage_num}')
    final_metrics = evaluate(
        model, val_loader,
        split_name=f'Val  [Stage {stage_num} FINAL]',
        show_confusion=True
    )

    # Save checkpoint
    save_checkpoint(model, optimizer, scheduler, stage_num, lora_name, final_metrics)

    return final_metrics


print('✅ train_one_stage() defined')

## 🚀 Cell 11 — Start Training (First Time)

Set `ACTIVE_CONFIG_IDX`:
- `0` → LoRA_A (r=8,  conservative baseline)
- `1` → LoRA_B (r=16, standard) ← **start here**
- `2` → LoRA_C (r=32, aggressive)

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect(); torch.cuda.empty_cache()

# ── Select LoRA config
ACTIVE_CONFIG_IDX = 1   # ← change to 0 or 2 for other configs
cfg       = LORA_CONFIGS[ACTIVE_CONFIG_IDX]
lora_name = cfg['name']

print(f'Config  : {lora_name}')
print(f'r       : {cfg["r"]}')
print(f'alpha   : {cfg["lora_alpha"]}')
print(f'lr      : {cfg["lr"]}')

# ── Build model (LLaMA via Unsloth + LoRA + Head)
model = build_model(cfg)

# ── Optimizer: only trainable params (LoRA adapters + classifier head)
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=cfg['lr'], weight_decay=WEIGHT_DECAY)

# ── LR Scheduler: cosine with warmup over ALL stages combined
steps_per_stage = (STAGE_SIZE // (BATCH_SIZE * GRAD_ACCUM)) * NUM_EPOCHS
total_steps     = steps_per_stage * NUM_STAGES
warmup_steps    = int(total_steps * WARMUP_RATIO)
scheduler       = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f'Total steps  : {total_steps}')
print(f'Warmup steps : {warmup_steps}')

# ── Run all stages
all_metrics = []
for stage_num in range(1, NUM_STAGES + 1):
    m = train_one_stage(
        model, stages[stage_num-1],
        optimizer, scheduler,
        stage_num, lora_name
    )
    all_metrics.append({'stage': stage_num, **m})

# ── Summary
print('\n' + '='*58)
print(f'  STAGED TRAINING COMPLETE — {lora_name}')
print('='*58)
print(f'  {"Stage":<8} {"Samples":>8} {"Accuracy":>10} {"F1 Macro":>10} {"F1 Weighted":>12}')
print('  ' + '-'*52)
for m in all_metrics:
    print(f'  {m["stage"]:<8} {m["stage"]*STAGE_SIZE:>8} '
          f'{m["accuracy"]:>10.4f} {m["f1_macro"]:>10.4f} {m["f1_weighted"]:>12.4f}')

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect(); torch.cuda.empty_cache()

# ── Select LoRA config
ACTIVE_CONFIG_IDX = 0   # ← change to 0 or 2 for other configs
cfg       = LORA_CONFIGS[ACTIVE_CONFIG_IDX]
lora_name = cfg['name']

print(f'Config  : {lora_name}')
print(f'r       : {cfg["r"]}')
print(f'alpha   : {cfg["lora_alpha"]}')
print(f'lr      : {cfg["lr"]}')

# ── Build model (LLaMA via Unsloth + LoRA + Head)
model = build_model(cfg)

# ── Optimizer: only trainable params (LoRA adapters + classifier head)
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=cfg['lr'], weight_decay=WEIGHT_DECAY)

# ── LR Scheduler: cosine with warmup over ALL stages combined
steps_per_stage = (STAGE_SIZE // (BATCH_SIZE * GRAD_ACCUM)) * NUM_EPOCHS
total_steps     = steps_per_stage * NUM_STAGES
warmup_steps    = int(total_steps * WARMUP_RATIO)
scheduler       = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f'Total steps  : {total_steps}')
print(f'Warmup steps : {warmup_steps}')

# ── Run all stages
all_metrics = []
for stage_num in range(1, NUM_STAGES + 1):
    m = train_one_stage(
        model, stages[stage_num-1],
        optimizer, scheduler,
        stage_num, lora_name
    )
    all_metrics.append({'stage': stage_num, **m})

# ── Summary
print('\n' + '='*58)
print(f'  STAGED TRAINING COMPLETE — {lora_name}')
print('='*58)
print(f'  {"Stage":<8} {"Samples":>8} {"Accuracy":>10} {"F1 Macro":>10} {"F1 Weighted":>12}')
print('  ' + '-'*52)
for m in all_metrics:
    print(f'  {m["stage"]:<8} {m["stage"]*STAGE_SIZE:>8} '
          f'{m["accuracy"]:>10.4f} {m["f1_macro"]:>10.4f} {m["f1_weighted"]:>12.4f}')

قعد 12 ساعه 50 دقيقه
Config  : LoRA_A
r       : 8
alpha   : 16
lr      : 0.0001
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.4 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.
trainable params: 6,815,744 || all params: 8,037,076,992 || trainable%: 0.0848
Total steps  : 1953
Warmup steps : 195

==========================================================
  STAGE 1 / 7  —  LoRA_A
  Samples : 3000 | Batches : 1500 | Epochs : 3
==========================================================
Unsloth: Will smartly offload gradients to save VRAM!
   Epoch 1 | Step  100/1500 | Loss: 3.8088 | Train Acc: 0.0250
   Epoch 1 | Step  200/1500 | Loss: 3.6048 | Train Acc: 0.0675
   Epoch 1 | Step  300/1500 | Loss: 3.4441 | Train Acc: 0.0783
   Epoch 1 | Step  400/1500 | Loss: 3.2913 | Train Acc: 0.1013
   Epoch 1 | Step  500/1500 | Loss: 3.1313 | Train Acc: 0.1210
   Epoch 1 | Step  600/1500 | Loss: 2.9986 | Train Acc: 0.1375
   Epoch 1 | Step  700/1500 | Loss: 2.8988 | Train Acc: 0.1457
   Epoch 1 | Step  800/1500 | Loss: 2.8220 | Train Acc: 0.1581
   Epoch 1 | Step  900/1500 | Loss: 2.7460 | Train Acc: 0.1711
   Epoch 1 | Step 1000/1500 | Loss: 2.6788 | Train Acc: 0.1840
   Epoch 1 | Step 1100/1500 | Loss: 2.6128 | Train Acc: 0.1977
   Epoch 1 | Step 1200/1500 | Loss: 2.5809 | Train Acc: 0.2062
   Epoch 1 | Step 1300/1500 | Loss: 2.5283 | Train Acc: 0.2115
   Epoch 1 | Step 1400/1500 | Loss: 2.4873 | Train Acc: 0.2200
   Epoch 1 | Step 1500/1500 | Loss: 2.4516 | Train Acc: 0.2280

  ── Epoch 1 done: loss=2.4516  train_acc=0.2280
==========================================================
  Val  [Stage 1 | Epoch 1]
==========================================================
  Accuracy    : 0.3285
  F1 Macro    : 0.1895
  F1 Weighted : 0.3019

                                 precision    recall  f1-score   support

           Billing and Payments       0.88      0.56      0.68       302
               Customer Service       0.22      0.45      0.30       448
                General Inquiry       0.00      0.00      0.00        42
                Human Resources       0.25      0.02      0.03        57
                     IT Support       0.15      0.19      0.17       350
                Product Support       0.33      0.12      0.18       554
          Returns and Exchanges       0.00      0.00      0.00       147
            Sales and Pre-Sales       0.11      0.01      0.02        88
Service Outages and Maintenance       0.50      0.03      0.07       115
              Technical Support       0.38      0.54      0.45       862

                       accuracy                           0.33      2965
                      macro avg       0.28      0.19      0.19      2965
                   weighted avg       0.34      0.33      0.30      2965

   Epoch 2 | Step  100/1500 | Loss: 1.9178 | Train Acc: 0.3350
   Epoch 2 | Step  200/1500 | Loss: 1.8595 | Train Acc: 0.3425
   Epoch 2 | Step  300/1500 | Loss: 1.8852 | Train Acc: 0.3433
   Epoch 2 | Step  400/1500 | Loss: 1.8464 | Train Acc: 0.3563
   Epoch 2 | Step  500/1500 | Loss: 1.8134 | Train Acc: 0.3660
   Epoch 2 | Step  600/1500 | Loss: 1.8442 | Train Acc: 0.3642
   Epoch 2 | Step  700/1500 | Loss: 1.8425 | Train Acc: 0.3607
   Epoch 2 | Step  800/1500 | Loss: 1.8468 | Train Acc: 0.3606
   Epoch 2 | Step  900/1500 | Loss: 1.8338 | Train Acc: 0.3700
   Epoch 2 | Step 1000/1500 | Loss: 1.8406 | Train Acc: 0.3635
   Epoch 2 | Step 1100/1500 | Loss: 1.8321 | Train Acc: 0.3641
   Epoch 2 | Step 1200/1500 | Loss: 1.8285 | Train Acc: 0.3658
   Epoch 2 | Step 1300/1500 | Loss: 1.8306 | Train Acc: 0.3623
   Epoch 2 | Step 1400/1500 | Loss: 1.8287 | Train Acc: 0.3604
   Epoch 2 | Step 1500/1500 | Loss: 1.8285 | Train Acc: 0.3593

  ── Epoch 2 done: loss=1.8285  train_acc=0.3593
==========================================================
  Val  [Stage 1 | Epoch 2]
==========================================================
  Accuracy    : 0.3717
  F1 Macro    : 0.2503
  F1 Weighted : 0.3402

                                 precision    recall  f1-score   support

           Billing and Payments       0.88      0.62      0.73       302
               Customer Service       0.26      0.18      0.21       448
                General Inquiry       0.00      0.00      0.00        42
                Human Resources       0.75      0.05      0.10        57
                     IT Support       0.29      0.02      0.04       350
                Product Support       0.25      0.52      0.34       554
          Returns and Exchanges       0.24      0.08      0.12       147
            Sales and Pre-Sales       0.00      0.00      0.00        88
Service Outages and Maintenance       0.69      0.37      0.49       115
              Technical Support       0.42      0.56      0.48       862

                       accuracy                           0.37      2965
                      macro avg       0.38      0.24      0.25      2965
                   weighted avg       0.39      0.37      0.34      2965

   Epoch 3 | Step  100/1500 | Loss: 1.4485 | Train Acc: 0.5050
   Epoch 3 | Step  200/1500 | Loss: 1.5326 | Train Acc: 0.4725
   Epoch 3 | Step  300/1500 | Loss: 1.4814 | Train Acc: 0.4900
   Epoch 3 | Step  400/1500 | Loss: 1.4940 | Train Acc: 0.4850
   Epoch 3 | Step  500/1500 | Loss: 1.4815 | Train Acc: 0.4860
   Epoch 3 | Step  600/1500 | Loss: 1.5074 | Train Acc: 0.4825
   Epoch 3 | Step  700/1500 | Loss: 1.5440 | Train Acc: 0.4700
   Epoch 3 | Step  800/1500 | Loss: 1.5496 | Train Acc: 0.4644
   Epoch 3 | Step  900/1500 | Loss: 1.5490 | Train Acc: 0.4606
   Epoch 3 | Step 1000/1500 | Loss: 1.5490 | Train Acc: 0.4615
   Epoch 3 | Step 1100/1500 | Loss: 1.5458 | Train Acc: 0.4655
   Epoch 3 | Step 1200/1500 | Loss: 1.5547 | Train Acc: 0.4604
   Epoch 3 | Step 1300/1500 | Loss: 1.5475 | Train Acc: 0.4619
   Epoch 3 | Step 1400/1500 | Loss: 1.5416 | Train Acc: 0.4664
   Epoch 3 | Step 1500/1500 | Loss: 1.5413 | Train Acc: 0.4640

  ── Epoch 3 done: loss=1.5413  train_acc=0.4640
==========================================================
  Val  [Stage 1 | Epoch 3]
==========================================================
  Accuracy    : 0.3929
  F1 Macro    : 0.3108
  F1 Weighted : 0.3813

                                 precision    recall  f1-score   support

           Billing and Payments       0.59      0.74      0.65       302
               Customer Service       0.26      0.24      0.25       448
                General Inquiry       0.00      0.00      0.00        42
                Human Resources       0.70      0.12      0.21        57
                     IT Support       0.29      0.29      0.29       350
                Product Support       0.31      0.36      0.34       554
          Returns and Exchanges       0.18      0.14      0.16       147
            Sales and Pre-Sales       0.18      0.09      0.12        88
Service Outages and Maintenance       0.67      0.56      0.61       115
              Technical Support       0.47      0.50      0.49       862

                       accuracy                           0.39      2965
                      macro avg       0.37      0.30      0.31      2965
                   weighted avg       0.38      0.39      0.38      2965


  ── Full evaluation at end of Stage 1
==========================================================
  Val  [Stage 1 FINAL]
==========================================================
  Accuracy    : 0.3929
  F1 Macro    : 0.3108
  F1 Weighted : 0.3813

                                 precision    recall  f1-score   support

           Billing and Payments       0.59      0.74      0.65       302
               Customer Service       0.26      0.24      0.25       448
                General Inquiry       0.00      0.00      0.00        42
                Human Resources       0.70      0.12      0.21        57
                     IT Support       0.29      0.29      0.29       350
                Product Support       0.31      0.36      0.34       554
          Returns and Exchanges       0.18      0.14      0.16       147
            Sales and Pre-Sales       0.18      0.09      0.12        88
Service Outages and Maintenance       0.67      0.56      0.61       115
              Technical Support       0.47      0.50      0.49       862

                       accuracy                           0.39      2965
                      macro avg       0.37      0.30      0.31      2965
                   weighted avg       0.38      0.39      0.38      2965


  ✅ Checkpoint saved → /content/drive/MyDrive/llama-cls-checkpoints/LoRA_A/stage_1
     acc=0.3929  F1-macro=0.3108

==========================================================
  STAGE 2 / 7  —  LoRA_A
  Samples : 3000 | Batches : 1500 | Epochs : 3
==========================================================
   Epoch 1 | Step  100/1500 | Loss: 1.6409 | Train Acc: 0.3700
   Epoch 1 | Step  200/1500 | Loss: 1.7073 | Train Acc: 0.3800
   Epoch 1 | Step  300/1500 | Loss: 1.7098 | Train Acc: 0.3817
   Epoch 1 | Step  400/1500 | Loss: 1.6927 | Train Acc: 0.3925
   Epoch 1 | Step  500/1500 | Loss: 1.7384 | Train Acc: 0.3840
   Epoch 1 | Step  600/1500 | Loss: 1.7286 | Train Acc: 0.3867
   Epoch 1 | Step  700/1500 | Loss: 1.7325 | Train Acc: 0.3821
   Epoch 1 | Step  800/1500 | Loss: 1.7289 | Train Acc: 0.3825
   Epoch 1 | Step  900/1500 | Loss: 1.7186 | Train Acc: 0.3806
   Epoch 1 | Step 1000/1500 | Loss: 1.7236 | Train Acc: 0.3765
   Epoch 1 | Step 1100/1500 | Loss: 1.7222 | Train Acc: 0.3750
   Epoch 1 | Step 1200/1500 | Loss: 1.7136 | Train Acc: 0.3779
   Epoch 1 | Step 1300/1500 | Loss: 1.7049 | Train Acc: 0.3819
   Epoch 1 | Step 1400/1500 | Loss: 1.6986 | Train Acc: 0.3829
   Epoch 1 | Step 1500/1500 | Loss: 1.6904 | Train Acc: 0.3833

  ── Epoch 1 done: loss=1.6904  train_acc=0.3833
==========================================================
  Val  [Stage 2 | Epoch 1]
==========================================================
  Accuracy    : 0.4165
  F1 Macro    : 0.3120
  F1 Weighted : 0.3822

                                 precision    recall  f1-score   support

           Billing and Payments       0.87      0.65      0.74       302
               Customer Service       0.33      0.23      0.27       448
                General Inquiry       0.00      0.00      0.00        42
                Human Resources       0.86      0.11      0.19        57
                     IT Support       0.61      0.04      0.08       350
                Product Support       0.30      0.43      0.35       554
          Returns and Exchanges       0.44      0.18      0.25       147
            Sales and Pre-Sales       0.46      0.07      0.12        88
Service Outages and Maintenance       0.89      0.47      0.61       115
              Technical Support       0.40      0.69      0.51       862

                       accuracy                           0.42      2965
                      macro avg       0.52      0.29      0.31      2965
                   weighted avg       0.47      0.42      0.38      2965

   Epoch 2 | Step  100/1500 | Loss: 1.3119 | Train Acc: 0.5600
   Epoch 2 | Step  200/1500 | Loss: 1.3471 | Train Acc: 0.5400
   Epoch 2 | Step  300/1500 | Loss: 1.3009 | Train Acc: 0.5433
   Epoch 2 | Step  400/1500 | Loss: 1.2740 | Train Acc: 0.5550
   Epoch 2 | Step  500/1500 | Loss: 1.2705 | Train Acc: 0.5450
   Epoch 2 | Step  600/1500 | Loss: 1.2850 | Train Acc: 0.5433
   Epoch 2 | Step  700/1500 | Loss: 1.2887 | Train Acc: 0.5364
   Epoch 2 | Step  800/1500 | Loss: 1.2901 | Train Acc: 0.5356
   Epoch 2 | Step  900/1500 | Loss: 1.2716 | Train Acc: 0.5411
   Epoch 2 | Step 1000/1500 | Loss: 1.2824 | Train Acc: 0.5360
   Epoch 2 | Step 1100/1500 | Loss: 1.2722 | Train Acc: 0.5409
   Epoch 2 | Step 1200/1500 | Loss: 1.2747 | Train Acc: 0.5396
   Epoch 2 | Step 1300/1500 | Loss: 1.2662 | Train Acc: 0.5419
   Epoch 2 | Step 1400/1500 | Loss: 1.2689 | Train Acc: 0.5425
   Epoch 2 | Step 1500/1500 | Loss: 1.2603 | Train Acc: 0.5460

  ── Epoch 2 done: loss=1.2603  train_acc=0.5460
==========================================================
  Val  [Stage 2 | Epoch 2]
==========================================================
  Accuracy    : 0.4384
  F1 Macro    : 0.3366
  F1 Weighted : 0.3987

                                 precision    recall  f1-score   support

           Billing and Payments       0.81      0.70      0.75       302
               Customer Service       0.38      0.19      0.25       448
                General Inquiry       0.50      0.02      0.05        42
                Human Resources       0.92      0.19      0.32        57
                     IT Support       0.46      0.14      0.22       350
                Product Support       0.37      0.32      0.35       554
          Returns and Exchanges       0.50      0.07      0.12       147
            Sales and Pre-Sales       0.37      0.11      0.17        88
Service Outages and Maintenance       0.62      0.62      0.62       115
              Technical Support       0.39      0.78      0.52       862

                       accuracy                           0.44      2965
                      macro avg       0.53      0.31      0.34      2965
                   weighted avg       0.46      0.44      0.40      2965

   Epoch 3 | Step  100/1500 | Loss: 0.8738 | Train Acc: 0.6850
   Epoch 3 | Step  200/1500 | Loss: 0.8518 | Train Acc: 0.7100
   Epoch 3 | Step  300/1500 | Loss: 0.8180 | Train Acc: 0.7133
   Epoch 3 | Step  400/1500 | Loss: 0.8097 | Train Acc: 0.7188
   Epoch 3 | Step  500/1500 | Loss: 0.8128 | Train Acc: 0.7140
   Epoch 3 | Step  600/1500 | Loss: 0.8146 | Train Acc: 0.7142
   Epoch 3 | Step  700/1500 | Loss: 0.7963 | Train Acc: 0.7250
   Epoch 3 | Step  800/1500 | Loss: 0.7955 | Train Acc: 0.7225
   Epoch 3 | Step  900/1500 | Loss: 0.8033 | Train Acc: 0.7217
   Epoch 3 | Step 1000/1500 | Loss: 0.8069 | Train Acc: 0.7230
   Epoch 3 | Step 1100/1500 | Loss: 0.8085 | Train Acc: 0.7214
   Epoch 3 | Step 1200/1500 | Loss: 0.8100 | Train Acc: 0.7229
   Epoch 3 | Step 1300/1500 | Loss: 0.8106 | Train Acc: 0.7231
   Epoch 3 | Step 1400/1500 | Loss: 0.8154 | Train Acc: 0.7232
   Epoch 3 | Step 1500/1500 | Loss: 0.8085 | Train Acc: 0.7277

  ── Epoch 3 done: loss=0.8085  train_acc=0.7277
==========================================================
  Val  [Stage 2 | Epoch 3]
==========================================================
  Accuracy    : 0.4648
  F1 Macro    : 0.4069
  F1 Weighted : 0.4469

                                 precision    recall  f1-score   support

           Billing and Payments       0.88      0.70      0.78       302
               Customer Service       0.39      0.26      0.31       448
                General Inquiry       0.71      0.12      0.20        42
                Human Resources       0.82      0.25      0.38        57
                     IT Support       0.36      0.26      0.31       350
                Product Support       0.41      0.35      0.38       554
          Returns and Exchanges       0.49      0.26      0.34       147
            Sales and Pre-Sales       0.67      0.11      0.19        88
Service Outages and Maintenance       0.73      0.57      0.64       115
              Technical Support       0.42      0.73      0.54       862

                       accuracy                           0.46      2965
                      macro avg       0.59      0.36      0.41      2965
                   weighted avg       0.49      0.46      0.45      2965


  ── Full evaluation at end of Stage 2
==========================================================
  Val  [Stage 2 FINAL]
==========================================================
  Accuracy    : 0.4648
  F1 Macro    : 0.4069
  F1 Weighted : 0.4469

                                 precision    recall  f1-score   support

           Billing and Payments       0.88      0.70      0.78       302
               Customer Service       0.39      0.26      0.31       448
                General Inquiry       0.71      0.12      0.20        42
                Human Resources       0.82      0.25      0.38        57
                     IT Support       0.36      0.26      0.31       350
                Product Support       0.41      0.35      0.38       554
          Returns and Exchanges       0.49      0.26      0.34       147
            Sales and Pre-Sales       0.67      0.11      0.19        88
Service Outages and Maintenance       0.73      0.57      0.64       115
              Technical Support       0.42      0.73      0.54       862

                       accuracy                           0.46      2965
                      macro avg       0.59      0.36      0.41      2965
                   weighted avg       0.49      0.46      0.45      2965


  ✅ Checkpoint saved → /content/drive/MyDrive/llama-cls-checkpoints/LoRA_A/stage_2
     acc=0.4648  F1-macro=0.4069

==========================================================
  STAGE 3 / 7  —  LoRA_A
  Samples : 3000 | Batches : 1500 | Epochs : 3
==========================================================
   Epoch 1 | Step  100/1500 | Loss: 1.6673 | Train Acc: 0.4500
   Epoch 1 | Step  200/1500 | Loss: 1.6483 | Train Acc: 0.4525
   Epoch 1 | Step  300/1500 | Loss: 1.6651 | Train Acc: 0.4433
   Epoch 1 | Step  400/1500 | Loss: 1.6799 | Train Acc: 0.4363
   Epoch 1 | Step  500/1500 | Loss: 1.6535 | Train Acc: 0.4460
   Epoch 1 | Step  600/1500 | Loss: 1.6402 | Train Acc: 0.4433
   Epoch 1 | Step  700/1500 | Loss: 1.6295 | Train Acc: 0.4443
   Epoch 1 | Step  800/1500 | Loss: 1.6150 | Train Acc: 0.4462
   Epoch 1 | Step  900/1500 | Loss: 1.6116 | Train Acc: 0.4478
   Epoch 1 | Step 1000/1500 | Loss: 1.6205 | Train Acc: 0.4405
   Epoch 1 | Step 1100/1500 | Loss: 1.6194 | Train Acc: 0.4382
   Epoch 1 | Step 1200/1500 | Loss: 1.6106 | Train Acc: 0.4383
   Epoch 1 | Step 1300/1500 | Loss: 1.5983 | Train Acc: 0.4404
   Epoch 1 | Step 1400/1500 | Loss: 1.5997 | Train Acc: 0.4411
   Epoch 1 | Step 1500/1500 | Loss: 1.5973 | Train Acc: 0.4427

  ── Epoch 1 done: loss=1.5973  train_acc=0.4427
==========================================================
  Val  [Stage 3 | Epoch 1]
==========================================================
  Accuracy    : 0.4627
  F1 Macro    : 0.3693
  F1 Weighted : 0.4327

                                 precision    recall  f1-score   support

           Billing and Payments       0.58      0.82      0.68       302
               Customer Service       0.35      0.35      0.35       448
                General Inquiry       0.40      0.05      0.09        42
                Human Resources       0.91      0.18      0.29        57
                     IT Support       0.45      0.16      0.24       350
                Product Support       0.43      0.34      0.38       554
          Returns and Exchanges       0.52      0.24      0.33       147
            Sales and Pre-Sales       0.69      0.10      0.18        88
Service Outages and Maintenance       0.61      0.61      0.61       115
              Technical Support       0.45      0.69      0.55       862

                       accuracy                           0.46      2965
                      macro avg       0.54      0.35      0.37      2965
                   weighted avg       0.47      0.46      0.43      2965

   Epoch 2 | Step  100/1500 | Loss: 1.1005 | Train Acc: 0.6550
   Epoch 2 | Step  200/1500 | Loss: 1.0536 | Train Acc: 0.6600
   Epoch 2 | Step  300/1500 | Loss: 1.0492 | Train Acc: 0.6600
   Epoch 2 | Step  400/1500 | Loss: 1.0317 | Train Acc: 0.6637
   Epoch 2 | Step  500/1500 | Loss: 1.0286 | Train Acc: 0.6600
   Epoch 2 | Step  600/1500 | Loss: 1.0321 | Train Acc: 0.6600
   Epoch 2 | Step  700/1500 | Loss: 1.0265 | Train Acc: 0.6593
   Epoch 2 | Step  800/1500 | Loss: 1.0237 | Train Acc: 0.6569
   Epoch 2 | Step  900/1500 | Loss: 1.0269 | Train Acc: 0.6572
   Epoch 2 | Step 1000/1500 | Loss: 1.0320 | Train Acc: 0.6540
   Epoch 2 | Step 1100/1500 | Loss: 1.0283 | Train Acc: 0.6559
   Epoch 2 | Step 1200/1500 | Loss: 1.0192 | Train Acc: 0.6600
   Epoch 2 | Step 1300/1500 | Loss: 1.0256 | Train Acc: 0.6565
   Epoch 2 | Step 1400/1500 | Loss: 1.0240 | Train Acc: 0.6593
   Epoch 2 | Step 1500/1500 | Loss: 1.0267 | Train Acc: 0.6570

  ── Epoch 2 done: loss=1.0267  train_acc=0.6570
==========================================================
  Val  [Stage 3 | Epoch 2]
==========================================================
  Accuracy    : 0.4769
  F1 Macro    : 0.3983
  F1 Weighted : 0.4470

                                 precision    recall  f1-score   support

           Billing and Payments       0.83      0.75      0.79       302
               Customer Service       0.42      0.32      0.36       448
                General Inquiry       0.20      0.02      0.04        42
                Human Resources       0.87      0.23      0.36        57
                     IT Support       0.49      0.16      0.24       350
                Product Support       0.48      0.29      0.36       554
          Returns and Exchanges       0.63      0.23      0.34       147
            Sales and Pre-Sales       0.59      0.23      0.33        88
Service Outages and Maintenance       0.78      0.51      0.62       115
              Technical Support       0.41      0.82      0.54       862

                       accuracy                           0.48      2965
                      macro avg       0.57      0.36      0.40      2965
                   weighted avg       0.51      0.48      0.45      2965

   Epoch 3 | Step  100/1500 | Loss: 0.5238 | Train Acc: 0.8300
   Epoch 3 | Step  200/1500 | Loss: 0.5190 | Train Acc: 0.8600
   Epoch 3 | Step  300/1500 | Loss: 0.5033 | Train Acc: 0.8633
   Epoch 3 | Step  400/1500 | Loss: 0.4925 | Train Acc: 0.8638
   Epoch 3 | Step  500/1500 | Loss: 0.5010 | Train Acc: 0.8590
   Epoch 3 | Step  600/1500 | Loss: 0.4995 | Train Acc: 0.8558
   Epoch 3 | Step  700/1500 | Loss: 0.5016 | Train Acc: 0.8579
   Epoch 3 | Step  800/1500 | Loss: 0.4977 | Train Acc: 0.8581
   Epoch 3 | Step  900/1500 | Loss: 0.4944 | Train Acc: 0.8578
   Epoch 3 | Step 1000/1500 | Loss: 0.4886 | Train Acc: 0.8615
   Epoch 3 | Step 1100/1500 | Loss: 0.4926 | Train Acc: 0.8591
   Epoch 3 | Step 1200/1500 | Loss: 0.4868 | Train Acc: 0.8633
   Epoch 3 | Step 1300/1500 | Loss: 0.4899 | Train Acc: 0.8592
   Epoch 3 | Step 1400/1500 | Loss: 0.4964 | Train Acc: 0.8568
   Epoch 3 | Step 1500/1500 | Loss: 0.4970 | Train Acc: 0.8550

  ── Epoch 3 done: loss=0.4970  train_acc=0.8550
==========================================================
  Val  [Stage 3 | Epoch 3]
==========================================================
  Accuracy    : 0.4712
  F1 Macro    : 0.4453
  F1 Weighted : 0.4715

                                 precision    recall  f1-score   support

           Billing and Payments       0.82      0.73      0.78       302
               Customer Service       0.38      0.40      0.39       448
                General Inquiry       0.33      0.05      0.08        42
                Human Resources       0.67      0.35      0.46        57
                     IT Support       0.35      0.31      0.33       350
                Product Support       0.36      0.58      0.44       554
          Returns and Exchanges       0.65      0.31      0.42       147
            Sales and Pre-Sales       0.47      0.28      0.35        88
Service Outages and Maintenance       0.81      0.63      0.71       115
              Technical Support       0.52      0.47      0.50       862

                       accuracy                           0.47      2965
                      macro avg       0.54      0.41      0.45      2965
                   weighted avg       0.50      0.47      0.47      2965


  ── Full evaluation at end of Stage 3
==========================================================
  Val  [Stage 3 FINAL]
==========================================================
  Accuracy    : 0.4712
  F1 Macro    : 0.4453
  F1 Weighted : 0.4715

                                 precision    recall  f1-score   support

           Billing and Payments       0.82      0.73      0.78       302
               Customer Service       0.38      0.40      0.39       448
                General Inquiry       0.33      0.05      0.08        42
                Human Resources       0.67      0.35      0.46        57
                     IT Support       0.35      0.31      0.33       350
                Product Support       0.36      0.58      0.44       554
          Returns and Exchanges       0.65      0.31      0.42       147
            Sales and Pre-Sales       0.47      0.28      0.35        88
Service Outages and Maintenance       0.81      0.63      0.71       115
              Technical Support       0.52      0.47      0.50       862

                       accuracy                           0.47      2965
                      macro avg       0.54      0.41      0.45      2965
                   weighted avg       0.50      0.47      0.47      2965


  ✅ Checkpoint saved → /content/drive/MyDrive/llama-cls-checkpoints/LoRA_A/stage_3
     acc=0.4712  F1-macro=0.4453

==========================================================
  STAGE 4 / 7  —  LoRA_A
  Samples : 3000 | Batches : 1500 | Epochs : 3
==========================================================
   Epoch 1 | Step  100/1500 | Loss: 1.5273 | Train Acc: 0.5250
   Epoch 1 | Step  200/1500 | Loss: 1.6460 | Train Acc: 0.4875
   Epoch 1 | Step  300/1500 | Loss: 1.5917 | Train Acc: 0.4983
   Epoch 1 | Step  400/1500 | Loss: 1.5784 | Train Acc: 0.4988
   Epoch 1 | Step  500/1500 | Loss: 1.5526 | Train Acc: 0.5010
   Epoch 1 | Step  600/1500 | Loss: 1.5217 | Train Acc: 0.5042
   Epoch 1 | Step  700/1500 | Loss: 1.5385 | Train Acc: 0.4971
   Epoch 1 | Step  800/1500 | Loss: 1.5382 | Train Acc: 0.4963
   Epoch 1 | Step  900/1500 | Loss: 1.5309 | Train Acc: 0.4956
   Epoch 1 | Step 1000/1500 | Loss: 1.5293 | Train Acc: 0.4970
   Epoch 1 | Step 1100/1500 | Loss: 1.5391 | Train Acc: 0.4945
   Epoch 1 | Step 1200/1500 | Loss: 1.5440 | Train Acc: 0.4921
   Epoch 1 | Step 1300/1500 | Loss: 1.5410 | Train Acc: 0.4900
   Epoch 1 | Step 1400/1500 | Loss: 1.5197 | Train Acc: 0.4936
   Epoch 1 | Step 1500/1500 | Loss: 1.5137 | Train Acc: 0.4897

  ── Epoch 1 done: loss=1.5137  train_acc=0.4897
==========================================================
  Val  [Stage 4 | Epoch 1]
==========================================================
  Accuracy    : 0.5076
  F1 Macro    : 0.4438
  F1 Weighted : 0.4893

                                 precision    recall  f1-score   support

           Billing and Payments       0.81      0.74      0.77       302
               Customer Service       0.42      0.34      0.38       448
                General Inquiry       0.75      0.07      0.13        42
                Human Resources       0.94      0.26      0.41        57
                     IT Support       0.49      0.28      0.36       350
                Product Support       0.47      0.39      0.43       554
          Returns and Exchanges       0.52      0.27      0.36       147
            Sales and Pre-Sales       0.83      0.22      0.34        88
Service Outages and Maintenance       0.79      0.61      0.69       115
              Technical Support       0.46      0.77      0.57       862

                       accuracy                           0.51      2965
                      macro avg       0.65      0.40      0.44      2965
                   weighted avg       0.54      0.51      0.49      2965

   Epoch 2 | Step  100/1500 | Loss: 1.0597 | Train Acc: 0.6550
   Epoch 2 | Step  200/1500 | Loss: 0.9846 | Train Acc: 0.6975
   Epoch 2 | Step  300/1500 | Loss: 0.9314 | Train Acc: 0.7217
   Epoch 2 | Step  400/1500 | Loss: 0.8789 | Train Acc: 0.7350
   Epoch 2 | Step  500/1500 | Loss: 0.8619 | Train Acc: 0.7400
   Epoch 2 | Step  600/1500 | Loss: 0.8626 | Train Acc: 0.7325
   Epoch 2 | Step  700/1500 | Loss: 0.8551 | Train Acc: 0.7336
   Epoch 2 | Step  800/1500 | Loss: 0.8585 | Train Acc: 0.7300
   Epoch 2 | Step  900/1500 | Loss: 0.8529 | Train Acc: 0.7333
   Epoch 2 | Step 1000/1500 | Loss: 0.8501 | Train Acc: 0.7345
   Epoch 2 | Step 1100/1500 | Loss: 0.8572 | Train Acc: 0.7305
   Epoch 2 | Step 1200/1500 | Loss: 0.8584 | Train Acc: 0.7279
   Epoch 2 | Step 1300/1500 | Loss: 0.8699 | Train Acc: 0.7235
   Epoch 2 | Step 1400/1500 | Loss: 0.8649 | Train Acc: 0.7243
   Epoch 2 | Step 1500/1500 | Loss: 0.8638 | Train Acc: 0.7270

  ── Epoch 2 done: loss=0.8638  train_acc=0.7270
==========================================================
  Val  [Stage 4 | Epoch 2]
==========================================================
  Accuracy    : 0.5292
  F1 Macro    : 0.4987
  F1 Weighted : 0.5267

                                 precision    recall  f1-score   support

           Billing and Payments       0.81      0.75      0.78       302
               Customer Service       0.44      0.46      0.45       448
                General Inquiry       0.38      0.12      0.18        42
                Human Resources       0.74      0.44      0.55        57
                     IT Support       0.46      0.38      0.41       350
                Product Support       0.45      0.52      0.48       554
          Returns and Exchanges       0.48      0.41      0.44       147
            Sales and Pre-Sales       0.77      0.34      0.47        88
Service Outages and Maintenance       0.62      0.66      0.64       115
              Technical Support       0.54      0.60      0.57       862

                       accuracy                           0.53      2965
                      macro avg       0.57      0.47      0.50      2965
                   weighted avg       0.54      0.53      0.53      2965

   Epoch 3 | Step  100/1500 | Loss: 0.3945 | Train Acc: 0.9400
   Epoch 3 | Step  200/1500 | Loss: 0.3735 | Train Acc: 0.9350
   Epoch 3 | Step  300/1500 | Loss: 0.3595 | Train Acc: 0.9350
   Epoch 3 | Step  400/1500 | Loss: 0.3600 | Train Acc: 0.9337
   Epoch 3 | Step  500/1500 | Loss: 0.3620 | Train Acc: 0.9310
   Epoch 3 | Step  600/1500 | Loss: 0.3616 | Train Acc: 0.9283
   Epoch 3 | Step  700/1500 | Loss: 0.3623 | Train Acc: 0.9271
   Epoch 3 | Step  800/1500 | Loss: 0.3651 | Train Acc: 0.9237
   Epoch 3 | Step  900/1500 | Loss: 0.3657 | Train Acc: 0.9222
   Epoch 3 | Step 1000/1500 | Loss: 0.3674 | Train Acc: 0.9190
   Epoch 3 | Step 1100/1500 | Loss: 0.3688 | Train Acc: 0.9168
   Epoch 3 | Step 1200/1500 | Loss: 0.3639 | Train Acc: 0.9175
   Epoch 3 | Step 1300/1500 | Loss: 0.3670 | Train Acc: 0.9154
   Epoch 3 | Step 1400/1500 | Loss: 0.3687 | Train Acc: 0.9146
   Epoch 3 | Step 1500/1500 | Loss: 0.3682 | Train Acc: 0.9150

  ── Epoch 3 done: loss=0.3682  train_acc=0.9150
==========================================================
  Val  [Stage 4 | Epoch 3]
==========================================================
  Accuracy    : 0.5261
  F1 Macro    : 0.4857
  F1 Weighted : 0.5195

                                 precision    recall  f1-score   support

           Billing and Payments       0.73      0.77      0.75       302
               Customer Service       0.46      0.43      0.45       448
                General Inquiry       0.28      0.17      0.21        42
                Human Resources       0.58      0.44      0.50        57
                     IT Support       0.45      0.34      0.39       350
                Product Support       0.48      0.50      0.49       554
          Returns and Exchanges       0.55      0.33      0.42       147
            Sales and Pre-Sales       0.52      0.40      0.45        88
Service Outages and Maintenance       0.62      0.64      0.63       115
              Technical Support       0.53      0.63      0.57       862

                       accuracy                           0.53      2965
                      macro avg       0.52      0.47      0.49      2965
                   weighted avg       0.52      0.53      0.52      2965


  ── Full evaluation at end of Stage 4
==========================================================
  Val  [Stage 4 FINAL]
==========================================================
  Accuracy    : 0.5261
  F1 Macro    : 0.4857
  F1 Weighted : 0.5195

                                 precision    recall  f1-score   support

           Billing and Payments       0.73      0.77      0.75       302
               Customer Service       0.46      0.43      0.45       448
                General Inquiry       0.28      0.17      0.21        42
                Human Resources       0.58      0.44      0.50        57
                     IT Support       0.45      0.34      0.39       350
                Product Support       0.48      0.50      0.49       554
          Returns and Exchanges       0.55      0.33      0.42       147
            Sales and Pre-Sales       0.52      0.40      0.45        88
Service Outages and Maintenance       0.62      0.64      0.63       115
              Technical Support       0.53      0.63      0.57       862

                       accuracy                           0.53      2965
                      macro avg       0.52      0.47      0.49      2965
                   weighted avg       0.52      0.53      0.52      2965


  ✅ Checkpoint saved → /content/drive/MyDrive/llama-cls-checkpoints/LoRA_A/stage_4
     acc=0.5261  F1-macro=0.4857

==========================================================
  STAGE 5 / 7  —  LoRA_A
  Samples : 3000 | Batches : 1500 | Epochs : 3
==========================================================
   Epoch 1 | Step  100/1500 | Loss: 1.4473 | Train Acc: 0.5550
   Epoch 1 | Step  200/1500 | Loss: 1.4497 | Train Acc: 0.5300
   Epoch 1 | Step  300/1500 | Loss: 1.4713 | Train Acc: 0.5150
   Epoch 1 | Step  400/1500 | Loss: 1.4799 | Train Acc: 0.5100
   Epoch 1 | Step  500/1500 | Loss: 1.4660 | Train Acc: 0.5200
   Epoch 1 | Step  600/1500 | Loss: 1.4645 | Train Acc: 0.5208
   Epoch 1 | Step  700/1500 | Loss: 1.4566 | Train Acc: 0.5200
   Epoch 1 | Step  800/1500 | Loss: 1.4510 | Train Acc: 0.5212
   Epoch 1 | Step  900/1500 | Loss: 1.4398 | Train Acc: 0.5233
   Epoch 1 | Step 1000/1500 | Loss: 1.4323 | Train Acc: 0.5245
   Epoch 1 | Step 1100/1500 | Loss: 1.4127 | Train Acc: 0.5277
   Epoch 1 | Step 1200/1500 | Loss: 1.4147 | Train Acc: 0.5292
   Epoch 1 | Step 1300/1500 | Loss: 1.4183 | Train Acc: 0.5296
   Epoch 1 | Step 1400/1500 | Loss: 1.4163 | Train Acc: 0.5296
   Epoch 1 | Step 1500/1500 | Loss: 1.4068 | Train Acc: 0.5323

  ── Epoch 1 done: loss=1.4068  train_acc=0.5323
==========================================================
  Val  [Stage 5 | Epoch 1]
==========================================================
  Accuracy    : 0.5440
  F1 Macro    : 0.5100
  F1 Weighted : 0.5381

                                 precision    recall  f1-score   support

           Billing and Payments       0.85      0.75      0.80       302
               Customer Service       0.50      0.43      0.46       448
                General Inquiry       0.44      0.17      0.24        42
                Human Resources       0.65      0.42      0.51        57
                     IT Support       0.55      0.29      0.38       350
                Product Support       0.43      0.60      0.50       554
          Returns and Exchanges       0.68      0.32      0.44       147
            Sales and Pre-Sales       0.48      0.47      0.47        88
Service Outages and Maintenance       0.84      0.62      0.71       115
              Technical Support       0.54      0.66      0.59       862

                       accuracy                           0.54      2965
                      macro avg       0.60      0.47      0.51      2965
                   weighted avg       0.56      0.54      0.54      2965

   Epoch 2 | Step  100/1500 | Loss: 0.9711 | Train Acc: 0.6850
   Epoch 2 | Step  200/1500 | Loss: 0.9268 | Train Acc: 0.6975
   Epoch 2 | Step  300/1500 | Loss: 0.8881 | Train Acc: 0.7167
   Epoch 2 | Step  400/1500 | Loss: 0.8850 | Train Acc: 0.7212
   Epoch 2 | Step  500/1500 | Loss: 0.8690 | Train Acc: 0.7230
   Epoch 2 | Step  600/1500 | Loss: 0.8460 | Train Acc: 0.7283
   Epoch 2 | Step  700/1500 | Loss: 0.8468 | Train Acc: 0.7307
   Epoch 2 | Step  800/1500 | Loss: 0.8432 | Train Acc: 0.7325
   Epoch 2 | Step  900/1500 | Loss: 0.8394 | Train Acc: 0.7344
   Epoch 2 | Step 1000/1500 | Loss: 0.8276 | Train Acc: 0.7410
   Epoch 2 | Step 1100/1500 | Loss: 0.8165 | Train Acc: 0.7445
   Epoch 2 | Step 1200/1500 | Loss: 0.8212 | Train Acc: 0.7412
   Epoch 2 | Step 1300/1500 | Loss: 0.8256 | Train Acc: 0.7404
   Epoch 2 | Step 1400/1500 | Loss: 0.8139 | Train Acc: 0.7457
   Epoch 2 | Step 1500/1500 | Loss: 0.8155 | Train Acc: 0.7457

  ── Epoch 2 done: loss=0.8155  train_acc=0.7457
==========================================================
  Val  [Stage 5 | Epoch 2]
==========================================================
  Accuracy    : 0.5467
  F1 Macro    : 0.5099
  F1 Weighted : 0.5422

                                 precision    recall  f1-score   support

           Billing and Payments       0.83      0.74      0.78       302
               Customer Service       0.42      0.57      0.49       448
                General Inquiry       0.45      0.12      0.19        42
                Human Resources       0.76      0.39      0.51        57
                     IT Support       0.51      0.30      0.38       350
                Product Support       0.50      0.50      0.50       554
          Returns and Exchanges       0.50      0.45      0.47       147
            Sales and Pre-Sales       0.61      0.42      0.50        88
Service Outages and Maintenance       0.75      0.63      0.69       115
              Technical Support       0.56      0.65      0.60       862

                       accuracy                           0.55      2965
                      macro avg       0.59      0.48      0.51      2965
                   weighted avg       0.56      0.55      0.54      2965

   Epoch 3 | Step  100/1500 | Loss: 0.4442 | Train Acc: 0.8800
   Epoch 3 | Step  200/1500 | Loss: 0.4117 | Train Acc: 0.8950
   Epoch 3 | Step  300/1500 | Loss: 0.4081 | Train Acc: 0.9050
   Epoch 3 | Step  400/1500 | Loss: 0.4156 | Train Acc: 0.9050
   Epoch 3 | Step  500/1500 | Loss: 0.4089 | Train Acc: 0.9070
   Epoch 3 | Step  600/1500 | Loss: 0.4119 | Train Acc: 0.9042
   Epoch 3 | Step  700/1500 | Loss: 0.4037 | Train Acc: 0.9050
   Epoch 3 | Step  800/1500 | Loss: 0.4082 | Train Acc: 0.9006
   Epoch 3 | Step  900/1500 | Loss: 0.4063 | Train Acc: 0.9006
   Epoch 3 | Step 1000/1500 | Loss: 0.4190 | Train Acc: 0.8970
   Epoch 3 | Step 1100/1500 | Loss: 0.4156 | Train Acc: 0.8986
   Epoch 3 | Step 1200/1500 | Loss: 0.4192 | Train Acc: 0.8971
   Epoch 3 | Step 1300/1500 | Loss: 0.4175 | Train Acc: 0.8977
   Epoch 3 | Step 1400/1500 | Loss: 0.4233 | Train Acc: 0.8946
   Epoch 3 | Step 1500/1500 | Loss: 0.4212 | Train Acc: 0.8940

  ── Epoch 3 done: loss=0.4212  train_acc=0.8940
==========================================================
  Val  [Stage 5 | Epoch 3]
==========================================================
  Accuracy    : 0.5390
  F1 Macro    : 0.5034
  F1 Weighted : 0.5353

                                 precision    recall  f1-score   support

           Billing and Payments       0.71      0.77      0.74       302
               Customer Service       0.45      0.49      0.47       448
                General Inquiry       0.44      0.17      0.24        42
                Human Resources       0.59      0.40      0.48        57
                     IT Support       0.45      0.39      0.42       350
                Product Support       0.48      0.53      0.50       554
          Returns and Exchanges       0.57      0.43      0.49       147
            Sales and Pre-Sales       0.51      0.34      0.41        88
Service Outages and Maintenance       0.73      0.67      0.70       115
              Technical Support       0.57      0.60      0.58       862

                       accuracy                           0.54      2965
                      macro avg       0.55      0.48      0.50      2965
                   weighted avg       0.54      0.54      0.54      2965


  ── Full evaluation at end of Stage 5
==========================================================
  Val  [Stage 5 FINAL]
==========================================================
  Accuracy    : 0.5390
  F1 Macro    : 0.5034
  F1 Weighted : 0.5353

                                 precision    recall  f1-score   support

           Billing and Payments       0.71      0.77      0.74       302
               Customer Service       0.45      0.49      0.47       448
                General Inquiry       0.44      0.17      0.24        42
                Human Resources       0.59      0.40      0.48        57
                     IT Support       0.45      0.39      0.42       350
                Product Support       0.48      0.53      0.50       554
          Returns and Exchanges       0.57      0.43      0.49       147
            Sales and Pre-Sales       0.51      0.34      0.41        88
Service Outages and Maintenance       0.73      0.67      0.70       115
              Technical Support       0.57      0.60      0.58       862

                       accuracy                           0.54      2965
                      macro avg       0.55      0.48      0.50      2965
                   weighted avg       0.54      0.54      0.54      2965




## ▶️ Cell 12 — Resume from Checkpoint (New Session)

**Use this cell when Colab disconnects.**
Run Cells 1→10 first, then run this cell.

Set `RESUME_LORA_NAME` and `RESUME_FROM_STAGE` (last **completed** stage).

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect(); torch.cuda.empty_cache()

# ════════════════════════════════════════════
# SET THESE TWO VALUES
# ════════════════════════════════════════════
RESUME_LORA_NAME  = 'LoRA_B'  # which config you were running
RESUME_FROM_STAGE = 2          # last COMPLETED stage number
# ════════════════════════════════════════════

cfg       = next(c for c in LORA_CONFIGS if c['name'] == RESUME_LORA_NAME)
lora_name = cfg['name']

# Rebuild model
model     = build_model(cfg)
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=cfg['lr'], weight_decay=WEIGHT_DECAY)

steps_per_stage = (STAGE_SIZE // (BATCH_SIZE * GRAD_ACCUM)) * NUM_EPOCHS
total_steps     = steps_per_stage * NUM_STAGES
warmup_steps    = int(total_steps * WARMUP_RATIO)
scheduler       = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# Load checkpoint
success = load_checkpoint(model, optimizer, scheduler, RESUME_FROM_STAGE, lora_name)

if success:
    print(f'\n✅ Resuming {lora_name} from Stage {RESUME_FROM_STAGE}')
    print(f'   Continuing: Stage {RESUME_FROM_STAGE+1} → {NUM_STAGES}')

    all_metrics = []
    for stage_num in range(RESUME_FROM_STAGE + 1, NUM_STAGES + 1):
        m = train_one_stage(
            model, stages[stage_num-1],
            optimizer, scheduler,
            stage_num, lora_name
        )
        all_metrics.append({'stage': stage_num, **m})

    print('\n✅ All remaining stages complete!')
else:
    print('❌ Checkpoint not found. Check RESUME_LORA_NAME and RESUME_FROM_STAGE.')

## 🧪 Cell 13 — Final Test Evaluation

In [ ]:
# Run after all stages are complete for a config
# Uses same test set as Classical ML for fair comparison

print(f'Final test evaluation — {lora_name}')
print(f'Test set size: {len(test_df)} (same split as Classical ML)')
print()

test_metrics = evaluate(
    model, test_loader,
    split_name=f'TEST SET — {lora_name} (all {NUM_STAGES} stages)',
    show_confusion=True
)

# Save to Drive
final_dir = os.path.join(CHECKPOINT_BASE, lora_name, 'final_test')
os.makedirs(final_dir, exist_ok=True)
with open(os.path.join(final_dir, 'test_metrics.json'), 'w') as f:
    json.dump({'lora_config': lora_name, **test_metrics}, f, indent=2)

print(f'\n✅ Test metrics saved to Drive')

## 📊 Cell 14 — LoRA HPT Comparison + Learning Curves

In [ ]:
# Run after completing ALL 3 LoRA configs

# ── 1. Final test comparison table
print('=' * 72)
print('  LORA HYPERPARAMETER TUNING — FINAL TEST RESULTS')
print('=' * 72)

rows = []
for cfg in LORA_CONFIGS:
    tf = os.path.join(CHECKPOINT_BASE, cfg['name'], 'final_test', 'test_metrics.json')
    if os.path.exists(tf):
        with open(tf) as f: m = json.load(f)
        rows.append({
            'Config': cfg['name'], 'r': cfg['r'],
            'lora_alpha': cfg['lora_alpha'], 'lr': cfg['lr'],
            'Accuracy': round(m['accuracy'], 4),
            'F1 Macro': round(m['f1_macro'], 4),
            'F1 Weighted': round(m['f1_weighted'], 4)
        })
    else:
        rows.append({'Config': cfg['name'], 'r': cfg['r'],
                     'lora_alpha': cfg['lora_alpha'], 'lr': cfg['lr'],
                     'Accuracy': 'N/A', 'F1 Macro': 'N/A', 'F1 Weighted': 'N/A'})

print(pd.DataFrame(rows).to_string(index=False))

# ── 2. Comparison with Classical ML
print('\n' + '=' * 72)
print('  COMPARISON WITH CLASSICAL ML')
print('=' * 72)
classical = pd.DataFrame([
    {'Model': 'Naive Bayes (BoW)',                'Accuracy': 0.4415, 'F1 Macro': 0.40},
    {'Model': 'Logistic Regression (TF-IDF)',     'Accuracy': 0.5089, 'F1 Macro': 0.51},
    {'Model': 'Logistic Regression (GridSearch)', 'Accuracy': 0.6378, 'F1 Macro': 0.64},
    {'Model': 'Random Forest (Word2Vec)',         'Accuracy': 0.6809, 'F1 Macro': 0.68},
])
print(classical.to_string(index=False))

# ── 3. Learning curves (stage-by-stage)
print('\n' + '=' * 72)
print('  LEARNING CURVES — ACCURACY PER STAGE')
print('=' * 72)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['steelblue', 'darkorange', 'green']

for ax_idx, metric_key in enumerate(['accuracy', 'f1_macro']):
    ax = axes[ax_idx]
    for i, cfg in enumerate(LORA_CONFIGS):
        xs, ys = [], []
        for s in range(1, NUM_STAGES + 1):
            mf = os.path.join(CHECKPOINT_BASE, cfg['name'], f'stage_{s}', 'metrics.json')
            if os.path.exists(mf):
                with open(mf) as f: m = json.load(f)
                xs.append(s * STAGE_SIZE)
                ys.append(m[metric_key])
        if xs:
            ax.plot(xs, ys, marker='o', label=cfg['name'],
                    color=colors[i], linewidth=2)

    metric_label = 'Accuracy' if metric_key == 'accuracy' else 'F1 Macro'
    ax.set_title(f'Learning Curve — {metric_label}', fontsize=13)
    ax.set_xlabel('Training Samples Seen')
    ax.set_ylabel(metric_label)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(
    os.path.join(CHECKPOINT_BASE, 'learning_curves.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()
print('✅ Learning curves saved to Drive')

## 🔍 Cell 15 — List All Checkpoints on Drive

In [ ]:
list_checkpoints()